➡ What this cell does:
Imports necessary modules and creates a tiny 8-sentence POS-tagged dataset.

In [3]:
import math, random

corpus = [
 "The/DT dog/NN barks/VBZ ./.",
 "A/DT cat/NN sleeps/VBZ ./.",
 "Mary/NNP sees/VBZ the/DT cat/NN ./.",
 "John/NNP will/MD run/VB tomorrow/NN ./.",
 "The/DT boy/NN eats/VBZ food/NN ./.",
 "They/PRP play/VBP football/NN outside/RB ./.",
 "Jane/NNP likes/VBZ Mary/NNP ./.",
 "Birds/NNS fly/VBP high/RB ./."
]

➡ What this cell does:
Turns each sentence into a list of (word, tag) pairs.

In [5]:
data = []
for line in corpus:
    sent = []
    for tok in line.split():
        w, t = tok.rsplit("/", 1)
        sent.append((w, t))
    data.append(sent)

data

[[('The', 'DT'), ('dog', 'NN'), ('barks', 'VBZ'), ('.', '.')],
 [('A', 'DT'), ('cat', 'NN'), ('sleeps', 'VBZ'), ('.', '.')],
 [('Mary', 'NNP'), ('sees', 'VBZ'), ('the', 'DT'), ('cat', 'NN'), ('.', '.')],
 [('John', 'NNP'),
  ('will', 'MD'),
  ('run', 'VB'),
  ('tomorrow', 'NN'),
  ('.', '.')],
 [('The', 'DT'), ('boy', 'NN'), ('eats', 'VBZ'), ('food', 'NN'), ('.', '.')],
 [('They', 'PRP'),
  ('play', 'VBP'),
  ('football', 'NN'),
  ('outside', 'RB'),
  ('.', '.')],
 [('Jane', 'NNP'), ('likes', 'VBZ'), ('Mary', 'NNP'), ('.', '.')],
 [('Birds', 'NNS'), ('fly', 'VBP'), ('high', 'RB'), ('.', '.')]]

➡ What this cell does:
Turns each sentence into a list of (word, tag) pairs.

In [ ]:
def make_folds(data, K=4, seed=1):
    random.seed(seed)
    idx = list(range(len(data)))
    random.shuffle(idx)

    groups = [idx[i::K] for i in range(K)]
    folds = []

    for i in range(K):
        test  = [data[j] for j in groups[i]]
        train = [data[j] for g in range(K) if g != i for j in groups[g]]
        folds.append((train, test))

    return folds


[([[('Jane', 'NNP'), ('likes', 'VBZ'), ('Mary', 'NNP'), ('.', '.')],
   [('The', 'DT'), ('dog', 'NN'), ('barks', 'VBZ'), ('.', '.')],
   [('A', 'DT'), ('cat', 'NN'), ('sleeps', 'VBZ'), ('.', '.')],
   [('The', 'DT'), ('boy', 'NN'), ('eats', 'VBZ'), ('food', 'NN'), ('.', '.')],
   [('They', 'PRP'),
    ('play', 'VBP'),
    ('football', 'NN'),
    ('outside', 'RB'),
    ('.', '.')],
   [('Mary', 'NNP'),
    ('sees', 'VBZ'),
    ('the', 'DT'),
    ('cat', 'NN'),
    ('.', '.')]],
  [[('John', 'NNP'),
    ('will', 'MD'),
    ('run', 'VB'),
    ('tomorrow', 'NN'),
    ('.', '.')],
   [('Birds', 'NNS'), ('fly', 'VBP'), ('high', 'RB'), ('.', '.')]]),
 ([[('John', 'NNP'),
    ('will', 'MD'),
    ('run', 'VB'),
    ('tomorrow', 'NN'),
    ('.', '.')],
   [('Birds', 'NNS'), ('fly', 'VBP'), ('high', 'RB'), ('.', '.')],
   [('A', 'DT'), ('cat', 'NN'), ('sleeps', 'VBZ'), ('.', '.')],
   [('The', 'DT'), ('boy', 'NN'), ('eats', 'VBZ'), ('food', 'NN'), ('.', '.')],
   [('They', 'PRP'),
    ('play', 'V

➡ What this cell does:
Counts how often each tag emits each word and transitions to the next tag.

In [4]:
def train_counts(train):
    START, END = "<S>", "</S>"
    E = {}   # E[tag][word]
    T = {}   # T[prev][next]
    C = {}   # tag counts
    V = set()

    for sent in train:
        prev = START
        for w, t in sent:
            E.setdefault(t, {})
            E[t][w] = E[t].get(w, 0) + 1

            T.setdefault(prev, {})
            T[prev][t] = T[prev].get(t, 0) + 1

            C[t] = C.get(t, 0) + 1
            prev = t
            V.add(w)

        T.setdefault(prev, {})
        T[prev][END] = T[prev].get(END, 0) + 1

    return E, T, C, V

➡ What this cell does:
Turns raw counts into log-probabilities so the Viterbi algorithm can use them.

In [5]:
def to_log(E, T, C, V):
    A = {}   # transition log-probs
    B = {}   # emission log-probs

    # transitions
    for prev, nxt in T.items():
        total = sum(nxt.values())
        denom = total + len(nxt)
        A[prev] = {t: math.log((nxt[t] + 1) / denom) for t in nxt}

    # emissions
    for tag, count in C.items():
        denom = count + len(V)
        B[tag] = {w: math.log((c + 1) / denom) for w, c in E[tag].items()}
        B[tag]["<UNK>"] = math.log(1 / denom)

    tags = sorted(C.keys())
    return A, B, tags

➡ What this cell does:
Implements the Viterbi algorithm to predict the best tag sequence for a sentence.

In [8]:
def viterbi(words, A, B, tags):
    START, END = "<S>", "</S>"
    n = len(words)

    # V[t][tag] = best log-score ending in this tag at position t
    V = [{t: -1e300 for t in tags} for _ in range(n)]
    back = [{} for _ in range(n)]

    # initialization
    for t in tags:
        trans = A.get(START, {}).get(t, math.log(1e-12))
        emit  = B[t].get(words[0], B[t]["<UNK>"])
        V[0][t] = trans + emit
        back[0][t] = START

    # recursion
    for i in range(1, n):
        for curr in tags:
            emit = B[curr].get(words[i], B[curr]["<UNK>"])
            best, bp = -1e300, None
            for prev in tags:
                trans = A.get(prev, {}).get(curr, math.log(1e-12))
                score = V[i-1][prev] + trans + emit
                if score > best:
                    best, bp = score, prev
            V[i][curr] = best
            back[i][curr] = bp

    # termination
    final = {t: V[-1][t] + A.get(t, {}).get(END, 0) for t in tags}
    last = max(final, key=final.get)

    # backtrack
    seq = [last]
    for i in range(n-1, 0, -1):
        seq.append(back[i][seq[-1]])

    return list(reversed(seq))

VITERBI INTO FORWARD ALGORITHM-> SLIGHT CHANGES REQUIRED ONLY

to convert this into forward algorithm,
change this max block

best, bp = -1e300, None
for prev in tags:
    trans = A.get(prev,{}).get(cur, math.log(1e-12))
    score = V[i-1][prev] + trans + emit
    if score > best:
        best, bp = score, prev
V[i][cur] = best
back[i][cur] = bp


into

scores = []
for prev in tags:
    trans = A.get(prev, {}).get(cur, math.log(1e-12))
    score = V[i-1][prev] + trans + emit
    scores.append(score)

# FORWARD algorithm uses SUM instead of MAX
V[i][cur] = logsumexp(scores)

# Forward algorithm does not track best path, so no backpointer
# back[i][cur] is NOT used


and add another helper function

def logsumexp(values):
    m = max(values)
    return m + math.log(sum(math.exp(v - m) for v in values))


➡ What this cell does:
Calculates a simple micro-F1 score to measure accuracy.

In [9]:
def micro_f1(true, pred):
    tp = fp = fn = 0
    for T, P in zip(true, pred):
        for a, b in zip(T, P):
            if a == b: tp += 1
            else: fp += 1; fn += 1
    return (2 * tp) / (2 * tp + fp + fn) if tp+fp+fn > 0 else 0


➡ What this cell does:
Runs training → decoding → evaluation for each fold and prints predictions + accuracy.

In [10]:
fold_data = make_folds(data, K=4)
scores = []

for i, (train, test) in enumerate(fold_data, 1):
    print("\n=== Fold", i, "===")

    E, T, C, V = train_counts(train)
    A, B, tags = to_log(E, T, C, V)

    true_all, pred_all = [], []

    for sent in test:
        words = [w for w,_ in sent]
        gold  = [t for _,t in sent]
        guess = viterbi(words, A, B, tags)

        print("Sentence:", " ".join(words))
        print("Gold    :", gold)
        print("Pred    :", guess, "\n")

        true_all.append(gold)
        pred_all.append(guess)

    f1 = micro_f1(true_all, pred_all)
    scores.append(f1)
    print("Fold micro-F1:", round(f1, 3))

print("\nAverage micro-F1:", round(sum(scores)/len(scores), 3))



=== Fold 1 ===
Sentence: John will run tomorrow .
Gold    : ['NNP', 'MD', 'VB', 'NN', '.']
Pred    : ['PRP', 'VBP', 'NN', 'RB', '.'] 

Sentence: Birds fly high .
Gold    : ['NNS', 'VBP', 'RB', '.']
Pred    : ['DT', 'NN', 'RB', '.'] 

Fold micro-F1: 0.333

=== Fold 2 ===
Sentence: Jane likes Mary .
Gold    : ['NNP', 'VBZ', 'NNP', '.']
Pred    : ['NNS', 'VBP', 'RB', '.'] 

Sentence: The dog barks .
Gold    : ['DT', 'NN', 'VBZ', '.']
Pred    : ['DT', 'NN', 'RB', '.'] 

Fold micro-F1: 0.5

=== Fold 3 ===
Sentence: A cat sleeps .
Gold    : ['DT', 'NN', 'VBZ', '.']
Pred    : ['DT', 'NN', 'RB', '.'] 

Sentence: The boy eats food .
Gold    : ['DT', 'NN', 'VBZ', 'NN', '.']
Pred    : ['NNP', 'MD', 'VB', 'NN', '.'] 

Fold micro-F1: 0.556

=== Fold 4 ===
Sentence: They play football outside .
Gold    : ['PRP', 'VBP', 'NN', 'RB', '.']
Pred    : ['NNP', 'MD', 'VB', 'NN', '.'] 

Sentence: Mary sees the cat .
Gold    : ['NNP', 'VBZ', 'DT', 'NN', '.']
Pred    : ['NNP', 'MD', 'VB', 'NN', '.'] 

Fold mi